# Plan sample size and minimum detectable effect

[All How-to Guides](README.md)

If you know the smallest effect that would justify a product change, estimate how many participants you need. If traffic is fixed, compute the effect that the available sample can detect. The example uses a continuous user-level metric, two independent groups, a two-sided significance level of 5%, and 80% power.

## 1. Use historical data to estimate variance

Use the historical metric to estimate variation in the future outcome. Take 500 users in each group. Assume a common variance for `MinSampleSize` and use group-specific variances for `MDEBySize`.

The generator defaults to `data_size=1_000`. The example spells it out to show the size of the input.

In [1]:
from scipy.stats import norm

from hypex.comparators import MDEBySize
from hypex.dataset import Dataset, TargetRole, TreatmentRole
from hypex.executor import MinSampleSize
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=1_000, dependent_division=False, random_state=42
)[["Target_lag_1"]]
frame = frame.assign(group=[0] * 500 + [1] * 500)
data = Dataset(
    roles={"group": TreatmentRole(), "Target_lag_1": TargetRole()},
    data=frame,
)
variance = float(data["Target_lag_1"].var())
{"users per group": frame["group"].value_counts().to_dict(), "variance": round(variance)}

{'users per group': {0: 500, 1: 500}, 'variance': 65865}

The example starts with a fixed sample size in each group. The estimated variances are inputs to the calculations below.

## 2. Calculate how many participants you need per group

Consider effects of 30 and 60 units. With equal group sizes and common variance $\sigma^2$, use the approximation $n\approx 2\sigma^2(z_{1-\alpha/2}+z_{1-\beta})^2/\Delta^2$. In this `MinSampleSize` interface, `power` takes the lower quantile level $\beta=0.20$. Here the example passes the quantiles directly.

`quantile_1` and `quantile_2` encode the significance level and power, so the constructor call can omit `alpha` and `power`. `equal_variance=True` selects the common-variance formula, and `variances=variance` supplies the historical estimate. If you omit `variances`, HypEx estimates variances from the input groups.

In [2]:
alpha = 0.05
desired_power = 0.80
beta = 1 - desired_power
users_per_group = {
    effect: MinSampleSize(
        mde=effect,
        equal_variance=True,
        variances=variance,
        quantile_1=float(norm.ppf(1 - alpha / 2)),
        quantile_2=float(norm.ppf(beta)),
    ).calc(data)["overall"]["min sample size"]
    for effect in (30.0, 60.0)
}
users_per_group

{30.0: 1149, 60.0: 288}

An effect of 30 requires 1,149 users per group. An effect of 60 requires 288. With the other inputs fixed, doubling the effect reduces the required sample by about a factor of four. Add the two group sizes for the experiment total.

## 3. Find the MDE for your available traffic

Now hold the sample size fixed. `MDEBySize` takes the desired power, 0.80, in `power`. `significance=0.95` corresponds to a two-sided significance level of 0.05.

In a direct `calc` call, `compare_by="groups"` compares groups. `target_fields_data` contains outcome values and `group_field_data` contains group labels. The example spells out `significance` and `power` to document the assumptions. Both match the method's defaults.

In [3]:
estimates = MDEBySize.calc(
    compare_by="groups",
    target_fields_data=data.select("Target_lag_1"),
    group_field_data=data.select("group"),
    significance=1 - alpha,
    power=desired_power,
)
mde = float(next(iter(estimates.values())).iget_values(row=0, column=0))
{"absolute MDE": round(mde, 2)}

{'absolute MDE': 45.37}

At 500 users per group, the minimum detectable effect is about 45.4 units. Detecting an effect of 30 under the same assumptions requires more participants.